# P22: Down syndrome fetal cortex Multiome — one-notebook workflow

**Canonical notebook:** `P22_down_syndrome_all_in_one.ipynb` (local Jupyter + Google Colab).

Dataset: Single-Cell Multiomic Atlas of Human Cortical Development in Down Syndrome
CELLxGENE collection `0e9fd1d3-ef4c-47c6-a2e4-ef4bfadf7c79`
CELLxGENE dataset `f16c25da-15bd-46a4-9a3f-17093f27a2f1` (expected 248,998 cells / 30 donors)
GEO: `GSE305153`

## Evidence boundary

- Safe default: live public metadata + synthetic wiring. **Not disease evidence.**
- Real H5AD / model work requires a **dated Professor Fang approval** and passing G1–G4.
- Gate numbering is canonical: **only G0–G9**.
- Do not call routing weights explanations or cross-attention.

Sources: [Nature Medicine](https://www.nature.com/articles/s41591-026-04211-1) · [PubMed](https://pubmed.ncbi.nlm.nih.gov/41545595/) · [CELLxGENE](https://cellxgene.cziscience.com/collections/0e9fd1d3-ef4c-47c6-a2e4-ef4bfadf7c79)


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("PYTHONDONTWRITEBYTECODE", "1")

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD
for candidate in (CWD, *CWD.parents):
    if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "p22").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from p22 import approval_blocked, load_approvals
from p22.data.adequacy import catalog_donor_frame, evaluate_donor_adequacy
from p22.data.catalog import EXPECTED_CELLS, EXPECTED_DONORS, run_catalog_preflight
from p22.data.group_splits import (
    aggregate_donor_probabilities,
    build_repeated_group_split_report,
)
from p22.data.resources import (
    PRIMARY_CAP,
    SENSITIVITY_CAPS,
    ResourceMeasurement,
    build_resource_report,
    decide_atac_branch,
)
from p22.eval.estimand import freeze_estimand
from p22.eval.faithfulness import INTERVENTIONS, intervention_table, run_all_interventions
from p22.eval.gates import ONE_NOTEBOOK_CONTRACT, empty_board, make_gate
from p22.eval.runtime import collect_runtime_report
from p22.eval.validation import build_validation_report
from p22.models.fusion import GatedFusionModel
from p22.models.named_baselines import (
    BaselineResult,
    baseline_table,
    blocked_baseline,
    chr21_dosage_baseline,
    covariate_logistic_baseline,
    majority_class_baseline,
    not_applicable_baseline,
    pseudobulk_rna_logistic,
)
from p22.reports.evidence_package import EvidencePackage
from p22.testing.synthetic import make_synthetic_multimodal

SEED = 20260728
MODEL_INIT_SEED = 0
DATASET_ID = "f16c25da-15bd-46a4-9a3f-17093f27a2f1"

approvals = load_approvals()
PROFESSOR_APPROVED = bool(approvals.get("approval_recorded")) and approvals.get("approval_state") == "approved"
RUN_REAL_DATA = os.getenv("P22_RUN_REAL_DATA", "0") == "1"
RUN_MODEL = os.getenv("P22_RUN_MODEL", "0") == "1"
if (RUN_REAL_DATA or RUN_MODEL) and not PROFESSOR_APPROVED:
    raise RuntimeError(
        "Approval gate: dated Professor Fang approval record required before real data/model."
    )

DATA_ROOT = Path(os.getenv("P22_DATA_ROOT", PROJECT_ROOT / "data" / "real"))
_base_output = Path(os.getenv("P22_OUTPUT_ROOT", PROJECT_ROOT / "reports" / "generated" / "verification" / "down_syndrome"))
from datetime import UTC, datetime
RUN_STAMP = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_ROOT = _base_output / f"run_{RUN_STAMP}"
SOURCE_NOTEBOOK = PROJECT_ROOT / "P22_down_syndrome_all_in_one.ipynb"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)



def safe_write_text(path: Path, text: str) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return path

board = empty_board()
print(ONE_NOTEBOOK_CONTRACT)
print({
    "python": sys.version.split()[0],
    "project_root": str(PROJECT_ROOT),
    "professor_approved": PROFESSOR_APPROVED,
    "approval_state": approvals.get("approval_state"),
    "run_real_data": RUN_REAL_DATA,
    "run_model": RUN_MODEL,
})


## Human-readable gates (G0–G9)

| Gate | Question |
|---|---|
| G0 | Can researcher run notebook locally and save outputs? |
| G1 | Are we using the intended public dataset? |
| G2 | Is dataset large enough at donor level? |
| G3 | Can local machine process real data within declared limits? |
| G4 | Is comparison defined before final training? |
| G5 | Can model be evaluated on unseen donors without leakage? |
| G6 | Does multimodal routing add stable value over simpler models? |
| G7 | Does prediction depend on routed modalities? |
| G8 | Are findings supported outside training data? |
| G9 | Can another researcher rerun and audit the result? |


In [ ]:
# G0 — Runtime
runtime = collect_runtime_report(
    project_root=PROJECT_ROOT,
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    source_notebook=SOURCE_NOTEBOOK,
)
board.set(make_gate(
    "G0",
    runtime.status,
    evidence=runtime.to_dict(),
    unknowns=runtime.open_questions,
    notes="output under reports/generated/; never overwrite source notebook",
))
safe_write_text(OUTPUT_ROOT / "local_runtime_report.json", json.dumps(runtime.to_dict(), indent=2) + "\n")
print(json.dumps(runtime.to_dict(), indent=2))
print("G0:", runtime.status)


## G1 — Live public metadata preflight

Catalog check only. No H5AD download.


In [ ]:
catalog = run_catalog_preflight()
board.set(make_gate(
    "G1",
    catalog.status,
    evidence=catalog.to_dict(),
    unknowns=catalog.open_questions,
    notes="same-nucleus pairing remains unknown until H5AD inspection",
))
safe_write_text(OUTPUT_ROOT / "preflight.json", json.dumps(catalog.to_dict(), indent=2) + "\n")
summary = catalog.summary or {}
print(json.dumps({
    "status": catalog.status,
    "dataset_id": summary.get("dataset_id"),
    "cells": summary.get("cell_count"),
    "donors": summary.get("donor_count"),
    "conditions": summary.get("conditions"),
    "condition_counts": summary.get("condition_counts_from_donor_suffix"),
    "h5ad_present": (summary.get("h5ad_asset") or {}).get("present"),
    "fragment_present": (summary.get("atac_fragment_asset") or {}).get("present"),
    "blocking_problems": list(catalog.blocking_problems),
    "open_questions": list(catalog.open_questions),
}, indent=2))
print("G1:", catalog.status)


## G2 — Donor adequacy thresholds

Frozen before matrix inspection: ≥15 donors/condition; ≥64 paired post-QC cells/donor; donor ratio ≤1.5:1; cell ratio ≤2:1; label purity 100%; ≥5 donor groups for folds.

Off-ramps if blocked: O1 scale-pool GSE305153; O2 RNA-only GSE280175; O3 reframe.


In [ ]:
donor_ids = list((catalog.summary or {}).get("donor_ids") or [])
donor_frame = catalog_donor_frame(donor_ids) if donor_ids else pd.DataFrame(columns=["donor_id", "condition"])
adequacy = evaluate_donor_adequacy(donor_frame, pairing_known=False)
board.set(make_gate(
    "G2",
    adequacy.status,
    evidence=adequacy.to_dict(),
    unknowns=adequacy.open_questions,
    notes="catalog donor counts only until post-QC H5AD",
))
print(json.dumps({
    "status": adequacy.status,
    "checks": adequacy.checks,
    "thresholds": adequacy.thresholds,
    "blocking_problems": list(adequacy.blocking_problems),
    "open_questions": list(adequacy.open_questions),
    "off_ramps": adequacy.off_ramp_options,
}, indent=2))
print("G2:", adequacy.status)


## G3 — Resource feasibility and ATAC B1/B2 branch

Primary cap 256 cells/donor; sensitivity caps 64 and 128. B1 peak block / B2a fragment build / B2b RNA-only reframe.


In [ ]:
fragment_present = bool(((catalog.summary or {}).get("atac_fragment_asset") or {}).get("present"))
h5ad_bytes = ((catalog.summary or {}).get("h5ad_asset") or {}).get("file_size")
atac_branch = decide_atac_branch(
    peak_block_present=None,
    fragment_asset_present=fragment_present,
    fragment_build_approved=False,
    resource_budget_ok=False,
)
measurements = [
    ResourceMeasurement(
        name="catalog_h5ad_size",
        cells_per_donor_cap=None,
        input_bytes=h5ad_bytes,
        disk_free_gb=runtime.disk_free_gb,
        notes="catalog metadata only; matrix not loaded",
    ),
    ResourceMeasurement(
        name="planned_primary_cap",
        cells_per_donor_cap=PRIMARY_CAP,
        notes="primary analysis cap frozen at 256; not yet measured on local H5AD",
    ),
    ResourceMeasurement(
        name="planned_sensitivity_caps",
        cells_per_donor_cap=None,
        notes=f"sensitivity caps {list(SENSITIVITY_CAPS)} after primary resource report",
    ),
]
resource = build_resource_report(
    measurements=measurements,
    atac_branch=atac_branch,
    h5ad_available=False,
    approval_present=PROFESSOR_APPROVED,
)
board.set(make_gate(
    "G3",
    resource.status,
    evidence=resource.to_dict(),
    unknowns=resource.open_questions,
    next_owner="Professor Fang" if not PROFESSOR_APPROVED else "researcher",
    notes="real H5AD load blocked without dated approval",
))
print(json.dumps(resource.to_dict(), indent=2))
print("G3:", resource.status)


## G4 — Frozen estimand

Label `condition == complete trisomy 21`; donor mean probability threshold 0.5; donor balanced accuracy; 5×5 StratifiedGroupKFold; model init seed 0; margin ≥0.07 for 15/15; validation `GSE280175` RNA-only.


In [ ]:
n_control = int((adequacy.checks or {}).get("n_control_donors") or 15)
n_ds = int((adequacy.checks or {}).get("n_ds_donors") or 15)
estimand_report = freeze_estimand(n_control_donors=n_control, n_ds_donors=n_ds)
board.set(make_gate(
    "G4",
    estimand_report.status,
    evidence=estimand_report.to_dict(),
    unknowns=estimand_report.open_questions,
    notes="protocol frozen before predictions",
))
print(json.dumps(estimand_report.to_dict(), indent=2))
assert estimand_report.estimand.margin >= 0.07
print("G4:", estimand_report.status, "margin=", estimand_report.estimand.margin)


## G5 — Donor-held-out leakage control

Synthetic wiring proves zero-overlap StratifiedGroupKFold. Real-data G5 stays blocked without approval.


In [ ]:
synth = make_synthetic_multimodal(
    n_donors=30,
    cells_per_donor=24,
    n_features_a=32,
    n_features_b=32,
    n_classes=2,
    seed=SEED,
)
# Disease wiring requires one condition per donor; override synthetic cell mixes.
pure_labels = np.asarray([0 if int(str(d).split('_')[1]) < 15 else 1 for d in synth.donor_ids])
synth_labels = pure_labels
split_report = build_repeated_group_split_report(
    synth.donor_ids,
    synth_labels,
    n_repeats=estimand_report.estimand.split_repeats,
    n_folds=estimand_report.estimand.folds_per_repeat,
    base_seed=SEED,
)
g5_status = "BLOCKED" if not PROFESSOR_APPROVED else split_report.status
board.set(make_gate(
    "G5",
    g5_status,
    evidence={
        "synthetic_split_status": split_report.status,
        "n_folds": len(split_report.folds),
        "donor_overlap_count": split_report.donor_overlap_count,
        "real_data": "BLOCKED" if not PROFESSOR_APPROVED else "eligible",
    },
    unknowns=("real H5AD not loaded",) if not PROFESSOR_APPROVED else (),
    notes="synthetic wiring only; not disease evidence",
))
print({
    "synthetic_split_status": split_report.status,
    "n_folds": len(split_report.folds),
    "donor_overlap_count": split_report.donor_overlap_count,
    "gate_status": g5_status,
})
assert split_report.donor_overlap_count == 0
print("G5:", g5_status)


## G6 — Named predictive baselines

Order: majority → chr21 → QC/covariate → pseudobulk RNA → RNA → ATAC → concat → gated.


In [ ]:
fold0 = split_report.folds[0]
train_donors = fold0.train_donors
test_donors = fold0.test_donors
y = synth_labels
donors = np.asarray(synth.donor_ids, dtype=object)
rna = synth.view_a
atac = synth.view_b
train_idx, test_idx = fold0.train_index, fold0.test_index

results = [
    majority_class_baseline(y, donors),
    chr21_dosage_baseline(None, y, donors, train_donors, test_donors),
    covariate_logistic_baseline(None, y, donors, train_donors, test_donors),
    pseudobulk_rna_logistic(rna, y, donors, train_donors, test_donors, n_features=16),
]

def donor_bal_acc(matrix, name):
    scaler = StandardScaler()
    x_train = scaler.fit_transform(matrix[train_idx])
    x_test = scaler.transform(matrix[test_idx])
    model = LogisticRegression(max_iter=400, class_weight="balanced", random_state=MODEL_INIT_SEED)
    model.fit(x_train, y[train_idx])
    proba = model.predict_proba(x_test)[:, 1]
    agg = aggregate_donor_probabilities(proba, donors[test_idx], threshold=0.5)
    truth = (
        pd.DataFrame({"donor_id": donors[test_idx], "label": y[test_idx]})
        .groupby("donor_id")["label"]
        .agg(lambda s: int(s.mode().iloc[0]))
    )
    merged = agg.merge(truth.rename("label"), on="donor_id")
    score = float(balanced_accuracy_score(merged["label"], merged["prediction"]))
    return BaselineResult(
        name=name,
        status="measured",
        donor_balanced_accuracy=score,
        n_donors=int(merged.shape[0]),
        n_cells=int(test_idx.size),
    )

results.append(donor_bal_acc(rna, "rna_only"))
if atac_branch.multimodal_claim_allowed:
    results.append(donor_bal_acc(atac, "atac_only"))
    results.append(donor_bal_acc(np.hstack([rna, atac]), "rna_atac_concat"))
    results.append(blocked_baseline("gated_fusion", "gated fusion blocked without approval"))
else:
    reason = f"ATAC branch {atac_branch.branch}; multimodal claim not allowed"
    results.append(not_applicable_baseline("atac_only", reason))
    results.append(not_applicable_baseline("rna_atac_concat", reason))
    results.append(not_applicable_baseline("gated_fusion", reason))

g6_status = "BLOCKED" if not PROFESSOR_APPROVED else "PASS"
baseline_rows = baseline_table(results)
board.set(make_gate(
    "G6",
    g6_status,
    evidence={"baselines": baseline_rows, "synthetic_wiring": True, "atac_branch": atac_branch.branch},
    unknowns=("chr21 annotation unavailable in synthetic mode", "real-data baselines not run"),
    notes="synthetic baseline wiring only; not disease evidence",
))
print(pd.DataFrame(baseline_rows)[["name", "status", "donor_balanced_accuracy", "not_applicable"]].to_string(index=False))
print("G6:", g6_status)


## G7 — Routing faithfulness interventions

Held-out ablation, clamp, uniform route, donor-preserving permutation. Routing signals ≠ explanations / ≠ cross-attention.


In [ ]:
n_features = 16
rna_scaler = StandardScaler().fit(rna[train_idx][:, :n_features])
atac_scaler = StandardScaler().fit(atac[train_idx][:, :n_features])
train_rna = rna_scaler.transform(rna[train_idx][:, :n_features])
test_rna = rna_scaler.transform(rna[test_idx][:, :n_features])
train_atac = atac_scaler.transform(atac[train_idx][:, :n_features])
test_atac = atac_scaler.transform(atac[test_idx][:, :n_features])

torch.manual_seed(MODEL_INIT_SEED)
gated = GatedFusionModel(
    n_features_a=n_features,
    n_features_b=n_features,
    n_classes=2,
    embed_dim=16,
    hidden_dim=32,
)
opt = torch.optim.Adam(gated.parameters(), lr=1e-2)
xa = torch.tensor(train_rna, dtype=torch.float32)
xb = torch.tensor(train_atac, dtype=torch.float32)
yt = torch.tensor(y[train_idx], dtype=torch.int64)
for _ in range(20):
    opt.zero_grad()
    out = gated(xa, xb)
    loss = torch.nn.functional.cross_entropy(out.logits, yt)
    loss.backward()
    opt.step()

effects = run_all_interventions(
    gated,
    views={"view_a": test_rna, "view_b": test_atac},
    labels=y[test_idx],
    donor_ids=donors[test_idx],
    train_views={"view_a": train_rna, "view_b": train_atac},
    metric_name="balanced_accuracy",
    seed=SEED,
)
intervention_rows = intervention_table(effects)
g7_status = "BLOCKED" if not PROFESSOR_APPROVED else "PASS"
board.set(make_gate(
    "G7",
    g7_status,
    evidence={"interventions": intervention_rows, "synthetic_wiring": True},
    unknowns=("real held-out routing not run",),
    notes="synthetic intervention plumbing only; routing signals ≠ explanations",
))
print(pd.DataFrame(intervention_rows).to_string(index=False))
print("expected interventions:", list(INTERVENTIONS))
print("G7:", g7_status)


## G8 — Independent biological validation

Frozen RNA-only resource: **GSE280175**. Not independent multimodal validation.


In [ ]:
validation = build_validation_report(
    marker_set=None,
    findings_available=False,
    approval_present=PROFESSOR_APPROVED,
)
board.set(make_gate(
    "G8",
    validation.status,
    evidence=validation.to_dict(),
    unknowns=validation.open_questions,
    next_owner="Professor Fang",
    notes=validation.claim_boundary,
))
print(json.dumps(validation.to_dict(), indent=2))
print("G8:", validation.status)


## G9 — Compact reproducible handoff package


In [ ]:
def git_head() -> str:
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
    except Exception:
        return "unknown"

run_id = f"safe_{SEED}"
package = EvidencePackage(
    run_dir=OUTPUT_ROOT / "runs" / run_id,
    manifest={
        "run_id": run_id,
        "commit": git_head(),
        "dataset_id": DATASET_ID,
        "expected_cells": EXPECTED_CELLS,
        "expected_donors": EXPECTED_DONORS,
        "catalog": catalog.to_dict(),
        "adequacy": adequacy.to_dict(),
        "resource": resource.to_dict(),
        "estimand": estimand_report.to_dict(),
        "gates": board.to_dict(),
        "approval_state": approvals.get("approval_state"),
        "professor_approved": PROFESSOR_APPROVED,
        "primary_cap": PRIMARY_CAP,
        "sensitivity_caps": list(SENSITIVITY_CAPS),
        "limitations": [
            "Safe-mode run: synthetic wiring and public metadata only.",
            "No disease-specific model result.",
            "Real H5AD / multimodal claims blocked without dated Professor Fang approval.",
            "GSE280175 is RNA-only validation, not multimodal.",
            "Routing weights are signals, not explanations or cross-attention.",
        ],
        "decision": "stop" if not PROFESSOR_APPROVED else "continue",
        "next_decision_owner": "Professor Fang",
        "next_decision": "record dated approval or reject condition-specific real-data work",
    },
    metrics_rows=baseline_rows,
    intervention_rows=intervention_rows,
    validation_rows=[item.to_dict() | {"gate_status": validation.status} for item in validation.resources],
)
paths = package.write()
board.set(make_gate(
    "G9",
    "PASS",
    evidence={"paths": paths, "decision": package.manifest["decision"]},
    notes="lean package under reports/generated/; source notebook not overwritten",
))

gate_df = pd.DataFrame(board.as_rows())
summary = {
    "gates": board.status_map(),
    "catalog_cells": (catalog.summary or {}).get("cell_count"),
    "catalog_donors": (catalog.summary or {}).get("donor_count"),
    "approval_state": approvals.get("approval_state"),
    "decision": package.manifest["decision"],
    "artifact_paths": paths,
    "claim_boundary": "No disease result in safe mode.",
}
safe_write_text(OUTPUT_ROOT / "notebook_summary.json", json.dumps(summary, indent=2) + "\n")
safe_write_text(OUTPUT_ROOT / "gate_board.json", json.dumps(board.to_dict(), indent=2) + "\n")
print(gate_df.to_string(index=False))
print(json.dumps(summary, indent=2))
assert board.get("G0").status in {"PASS", "INCONCLUSIVE"}
assert board.get("G9").status == "PASS"
assert SOURCE_NOTEBOOK.is_file()
